# 100 - Advanced Databricks Billing and FinOps Analysis

This notebook builds a production-oriented cost-analysis layer from Databricks system tables. It goes beyond simple DBU totals and covers effective-dated pricing, corrections, attribution, chargeback, unit economics, anomalies, idle resources, governance, and operationalization.

## Learning objectives

- Understand the grain and limitations of `system.billing.usage` and `system.billing.list_prices`.
- Calculate correction-aware estimated list cost using the price effective at usage time.
- Attribute cost to workspace, identity, product, SKU, jobs, pipelines, warehouses, notebooks, and tags.
- Measure coverage gaps so unallocated cost is visible rather than silently lost.
- Detect trends, anomalies, unit-cost regressions, and optimization opportunities.
- Design reusable Gold tables, dashboards, alerts, budgets, and chargeback controls.

> **Important:** Estimated list cost is not an invoice. Contract discounts, credits, commitments, taxes, allowances, marketplace charges, and cloud infrastructure costs can differ. Keep currencies and usage units separate. Billing timestamps are UTC.

## 0. Architecture and security

The billing tables are account-wide and may reveal identities, resource names, usage patterns, and tags. Grant access through a least-privilege group; do not export raw system-table data unnecessarily. A typical design is:

`system.billing.*` -> governed correction-aware view -> daily Gold aggregates -> dashboard / SQL alert / budget process

The core tables are global. Some enrichment tables such as query history and compute events are regional and might require an administrator to enable their schemas.

In [ ]:
%python
# Change these values in the widget bar and rerun the SQL sections.
dbutils.widgets.removeAll()
dbutils.widgets.text('days_back', '90', 'Days of history')
dbutils.widgets.text('workspace_filter', 'ALL', 'Workspace ID or ALL')
dbutils.widgets.text('tag_key', 'project', 'Allocation tag key')
dbutils.widgets.text('daily_cost_alert', '1000', 'Daily cost alert threshold')

## 1. Access, availability, and data freshness

Start with discovery. Do not assume that every account has the same products, units, metadata fields, or optional system schemas.

In [ ]:
%sql
SHOW TABLES IN system.billing;

In [ ]:
%sql
DESCRIBE TABLE system.billing.usage;

In [ ]:
%sql
SELECT
  min(usage_date) AS earliest_usage_date,
  max(usage_date) AS latest_usage_date,
  max(ingestion_date) AS latest_ingestion_date,
  datediff(current_date(), max(usage_date)) AS usage_lag_days,
  count(*) AS physical_records,
  count(DISTINCT record_id) AS distinct_record_ids
FROM system.billing.usage;

### Discover measurements before aggregating

`usage_quantity` has meaning only together with `usage_unit` and `usage_type`. Adding DBUs, tokens, bytes, GPU time, and storage quantities produces a meaningless number.

In [ ]:
%sql
SELECT
  usage_type, usage_unit, billing_origin_product, sku_name,
  round(sum(usage_quantity), 4) AS net_quantity,
  count(*) AS records
FROM system.billing.usage
WHERE usage_date >= date_sub(current_date(), CAST('${days_back}' AS INT))
  AND ('${workspace_filter}' = 'ALL' OR CAST(workspace_id AS STRING) = '${workspace_filter}')
GROUP BY ALL
ORDER BY usage_type, usage_unit, net_quantity DESC;

## 2. Corrections and record grain

Corrections are represented by signed records: `RETRACTION` negates incorrect usage and `RESTATEMENT` supplies corrected values. Never filter corrections out of a cost calculation. Sum all record types and remove only groups whose net quantity is zero.

In [ ]:
%sql
SELECT
  record_type,
  count(*) AS records,
  round(sum(usage_quantity), 6) AS signed_quantity,
  min(usage_date) AS first_date,
  max(usage_date) AS last_date
FROM system.billing.usage
WHERE usage_date >= date_sub(current_date(), CAST('${days_back}' AS INT))
GROUP BY record_type
ORDER BY record_type;

In [ ]:
%sql
-- Inspect recent correction chains without assuming record_id is unique across the chain.
SELECT
  ingestion_date, usage_date, record_id, record_type, sku_name,
  usage_quantity, usage_unit, usage_metadata, identity_metadata
FROM system.billing.usage
WHERE record_type <> 'ORIGINAL'
ORDER BY ingestion_date DESC, record_id, record_type
LIMIT 100;

## 3. Build one reusable, effective-dated cost view

The price join must match SKU and cloud and must select the price interval valid when usage occurred. A left join preserves usage with missing price coverage so that it can be measured. The `pricing.effective_list.default` field represents effective list price; it still does not represent a negotiated invoice rate.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW billing_cost_enriched AS
SELECT
  u.record_id, u.account_id, u.workspace_id,
  u.usage_start_time, u.usage_end_time, u.usage_date, u.ingestion_date,
  u.sku_name, u.cloud, u.usage_type, u.usage_unit, u.usage_quantity,
  u.record_type, u.billing_origin_product,
  u.usage_metadata, u.identity_metadata, u.custom_tags, u.product_features,
  p.currency_code, p.pricing.effective_list.default AS effective_list_price,
  u.usage_quantity * p.pricing.effective_list.default AS estimated_list_cost,
  CASE WHEN p.sku_name IS NULL THEN true ELSE false END AS missing_price
FROM system.billing.usage u
LEFT JOIN system.billing.list_prices p
  ON u.sku_name = p.sku_name
 AND u.cloud = p.cloud
 AND u.usage_start_time >= p.price_start_time
 AND (u.usage_end_time < p.price_end_time OR p.price_end_time IS NULL)
WHERE u.usage_date >= date_sub(current_date(), CAST('${days_back}' AS INT))
  AND ('${workspace_filter}' = 'ALL' OR CAST(u.workspace_id AS STRING) = '${workspace_filter}');

In [ ]:
%sql
-- Pricing coverage is a required data-quality check.
SELECT
  sku_name, cloud, usage_unit,
  count(*) AS usage_records,
  count_if(missing_price) AS unpriced_records,
  round(100.0 * count_if(missing_price) / count(*), 2) AS unpriced_record_pct,
  round(sum(CASE WHEN missing_price THEN usage_quantity ELSE 0 END), 6) AS unpriced_quantity
FROM billing_cost_enriched
GROUP BY ALL
HAVING unpriced_records > 0
ORDER BY unpriced_records DESC;

## 4. Executive cost overview

Keep currency as a grouping key. Never add USD, EUR, GBP, or another currency without an explicit governed FX conversion table.

In [ ]:
%sql
SELECT
  currency_code,
  round(sum(estimated_list_cost), 2) AS estimated_list_cost,
  round(sum(CASE WHEN usage_date >= date_trunc('MONTH', current_date()) THEN estimated_list_cost ELSE 0 END), 2) AS month_to_date_cost,
  count(DISTINCT workspace_id) AS workspaces,
  count(DISTINCT sku_name) AS skus,
  max(usage_date) AS latest_usage_date
FROM billing_cost_enriched
GROUP BY currency_code
ORDER BY estimated_list_cost DESC;

In [ ]:
%sql
SELECT
  usage_date, currency_code,
  round(sum(estimated_list_cost), 2) AS daily_cost,
  round(sum(sum(estimated_list_cost)) OVER (PARTITION BY currency_code ORDER BY usage_date), 2) AS cumulative_cost
FROM billing_cost_enriched
GROUP BY usage_date, currency_code
ORDER BY usage_date, currency_code;

In [ ]:
%sql
SELECT
  coalesce(billing_origin_product, 'UNATTRIBUTED') AS product,
  sku_name, currency_code,
  product_features.is_serverless AS is_serverless,
  product_features.is_photon AS is_photon,
  product_features.performance_target AS performance_target,
  round(sum(estimated_list_cost), 2) AS estimated_list_cost,
  round(100.0 * sum(estimated_list_cost) / sum(sum(estimated_list_cost)) OVER (PARTITION BY currency_code), 2) AS currency_cost_pct
FROM billing_cost_enriched
GROUP BY ALL
ORDER BY estimated_list_cost DESC;

## 5. Workspace, identity, and resource attribution

Attribution is not universally populated. Jobs on dedicated job compute or serverless compute provide better job/run attribution than jobs sharing all-purpose compute. Measure null coverage before promising showback or chargeback accuracy.

In [ ]:
%sql
SELECT
  currency_code,
  round(sum(estimated_list_cost), 2) AS total_cost,
  round(sum(CASE WHEN identity_metadata.run_as IS NULL THEN estimated_list_cost ELSE 0 END), 2) AS no_run_as_cost,
  round(sum(CASE WHEN usage_metadata.job_id IS NULL THEN estimated_list_cost ELSE 0 END), 2) AS no_job_id_cost,
  round(sum(CASE WHEN usage_metadata.cluster_id IS NULL THEN estimated_list_cost ELSE 0 END), 2) AS no_cluster_id_cost,
  round(sum(CASE WHEN usage_metadata.warehouse_id IS NULL THEN estimated_list_cost ELSE 0 END), 2) AS no_warehouse_id_cost
FROM billing_cost_enriched
GROUP BY currency_code;

In [ ]:
%sql
SELECT
  workspace_id,
  coalesce(identity_metadata.run_as, identity_metadata.owned_by, 'UNATTRIBUTED') AS responsible_identity,
  currency_code,
  round(sum(estimated_list_cost), 2) AS estimated_list_cost
FROM billing_cost_enriched
GROUP BY ALL
ORDER BY estimated_list_cost DESC
LIMIT 100;

In [ ]:
%sql
-- One resource-key view supports investigation across workload types.
SELECT
  workspace_id, billing_origin_product, currency_code,
  coalesce(
    usage_metadata.job_run_id, usage_metadata.job_id,
    usage_metadata.dlt_pipeline_id, usage_metadata.warehouse_id,
    usage_metadata.cluster_id, usage_metadata.notebook_id, 'UNATTRIBUTED'
  ) AS resource_key,
  round(sum(estimated_list_cost), 2) AS estimated_list_cost
FROM billing_cost_enriched
GROUP BY ALL
ORDER BY estimated_list_cost DESC
LIMIT 100;

## 6. Job and pipeline costs

The first query works from billing alone. The second, optional query enriches job IDs with slowly changing job names from `system.lakeflow.jobs`. It may require an administrator to enable the Lakeflow system schema.

In [ ]:
%sql
SELECT
  workspace_id, usage_metadata.job_id AS job_id,
  usage_metadata.job_run_id AS job_run_id, currency_code,
  min(usage_start_time) AS billed_from, max(usage_end_time) AS billed_to,
  round(sum(estimated_list_cost), 2) AS estimated_list_cost
FROM billing_cost_enriched
WHERE billing_origin_product = 'JOBS'
GROUP BY ALL
ORDER BY estimated_list_cost DESC
LIMIT 100;

In [ ]:
%sql
-- Optional enrichment: latest known job name.
WITH latest_jobs AS (
  SELECT workspace_id, job_id, name
  FROM system.lakeflow.jobs
  QUALIFY row_number() OVER (PARTITION BY workspace_id, job_id ORDER BY change_time DESC) = 1
), job_cost AS (
  SELECT workspace_id, usage_metadata.job_id AS job_id, currency_code,
         sum(estimated_list_cost) AS estimated_list_cost
  FROM billing_cost_enriched
  WHERE billing_origin_product = 'JOBS' AND usage_metadata.job_id IS NOT NULL
  GROUP BY ALL
)
SELECT c.workspace_id, c.job_id, coalesce(j.name, 'UNKNOWN_OR_DELETED') AS job_name,
       c.currency_code, round(c.estimated_list_cost, 2) AS estimated_list_cost
FROM job_cost c LEFT JOIN latest_jobs j USING (workspace_id, job_id)
ORDER BY estimated_list_cost DESC;

In [ ]:
%sql
SELECT
  workspace_id, usage_metadata.dlt_pipeline_id AS pipeline_id,
  product_features.dlt_tier AS pipeline_tier,
  product_features.is_serverless AS is_serverless,
  product_features.performance_target AS performance_target,
  currency_code, round(sum(estimated_list_cost), 2) AS estimated_list_cost
FROM billing_cost_enriched
WHERE billing_origin_product = 'DLT' OR usage_metadata.dlt_pipeline_id IS NOT NULL
GROUP BY ALL
ORDER BY estimated_list_cost DESC;

## 7. SQL warehouse cost and utilization

Billing gives warehouse cost. `system.query.history`, `system.compute.warehouses`, and `system.compute.warehouse_events` add workload and lifecycle evidence. These optional regional tables may not be enabled. Cost per query is a useful indicator, but evenly allocating a warehouse's cost across queries is an approximation because queries overlap and warehouses can be idle.

In [ ]:
%sql
WITH warehouse_cost AS (
  SELECT workspace_id, usage_metadata.warehouse_id AS warehouse_id, usage_date, currency_code,
         sum(estimated_list_cost) AS daily_cost
  FROM billing_cost_enriched
  WHERE usage_metadata.warehouse_id IS NOT NULL
  GROUP BY ALL
), query_activity AS (
  SELECT workspace_id, compute.warehouse_id AS warehouse_id, CAST(start_time AS DATE) AS usage_date,
         count(*) AS query_count,
         sum(total_duration_ms) / 1000.0 AS total_query_seconds,
         sum(waiting_at_capacity_duration_ms) / 1000.0 AS capacity_wait_seconds
  FROM system.query.history
  WHERE start_time >= date_sub(current_date(), CAST('${days_back}' AS INT))
  GROUP BY ALL
)
SELECT c.*, coalesce(q.query_count, 0) AS query_count,
       round(q.total_query_seconds, 2) AS total_query_seconds,
       round(q.capacity_wait_seconds, 2) AS capacity_wait_seconds,
       round(c.daily_cost / nullif(q.query_count, 0), 4) AS approximate_cost_per_query
FROM warehouse_cost c LEFT JOIN query_activity q USING (workspace_id, warehouse_id, usage_date)
ORDER BY c.daily_cost DESC;

## 8. Tags, allocation coverage, and chargeback

Tags should represent governed allocation dimensions such as cost center, business unit, environment, product, and owner. First discover keys; then calculate tagged and untagged cost. A production chargeback model normally adds an allocation mapping table for shared cost and a policy version/effective date.

In [ ]:
%sql
SELECT tag_key, count(*) AS records
FROM billing_cost_enriched
LATERAL VIEW explode(map_keys(custom_tags)) e AS tag_key
GROUP BY tag_key
ORDER BY records DESC;

In [ ]:
%sql
SELECT
  coalesce(element_at(custom_tags, '${tag_key}'), 'UNTAGGED') AS allocation_value,
  currency_code,
  round(sum(estimated_list_cost), 2) AS estimated_list_cost,
  round(100.0 * sum(estimated_list_cost) / sum(sum(estimated_list_cost)) OVER (PARTITION BY currency_code), 2) AS currency_cost_pct
FROM billing_cost_enriched
GROUP BY ALL
ORDER BY estimated_list_cost DESC;

In [ ]:
%sql
-- KPI for a tag-compliance dashboard or policy alert.
SELECT currency_code,
  round(sum(estimated_list_cost), 2) AS total_cost,
  round(sum(CASE WHEN element_at(custom_tags, '${tag_key}') IS NULL THEN estimated_list_cost ELSE 0 END), 2) AS untagged_cost,
  round(100.0 * sum(CASE WHEN element_at(custom_tags, '${tag_key}') IS NULL THEN estimated_list_cost ELSE 0 END)
        / nullif(sum(estimated_list_cost), 0), 2) AS untagged_cost_pct
FROM billing_cost_enriched
GROUP BY currency_code;

## 9. Trend, forecast, and anomaly analysis

The following statistics are teaching baselines, not complete forecasting models. Use complete days/months for comparisons and evaluate anomalies separately per currency and product.

In [ ]:
%sql
WITH daily AS (
  SELECT usage_date, currency_code, sum(estimated_list_cost) AS daily_cost
  FROM billing_cost_enriched
  WHERE usage_date < current_date()
  GROUP BY ALL
), scored AS (
  SELECT *,
    avg(daily_cost) OVER (PARTITION BY currency_code ORDER BY usage_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS prior_7d_avg,
    stddev_samp(daily_cost) OVER (PARTITION BY currency_code ORDER BY usage_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS prior_7d_stddev
  FROM daily
)
SELECT *,
  round((daily_cost - prior_7d_avg) / nullif(prior_7d_stddev, 0), 2) AS z_score,
  CASE WHEN daily_cost > prior_7d_avg + 3 * prior_7d_stddev THEN 'HIGH_ANOMALY' ELSE 'NORMAL' END AS status
FROM scored
ORDER BY usage_date DESC, currency_code;

In [ ]:
%sql
-- Month-to-date run-rate forecast. This assumes future days resemble elapsed days.
SELECT currency_code,
  round(sum(estimated_list_cost), 2) AS month_to_date_cost,
  day(current_date()) AS elapsed_calendar_days,
  day(last_day(current_date())) AS days_in_month,
  round(sum(estimated_list_cost) / day(current_date()) * day(last_day(current_date())), 2) AS simple_month_forecast
FROM billing_cost_enriched
WHERE usage_date >= date_trunc('MONTH', current_date())
GROUP BY currency_code;

In [ ]:
%sql
-- Compare the two previous complete months by product.
WITH monthly AS (
  SELECT date_trunc('MONTH', usage_date) AS month_start, currency_code,
         coalesce(billing_origin_product, 'UNATTRIBUTED') AS product,
         sum(estimated_list_cost) AS cost
  FROM billing_cost_enriched
  WHERE usage_date >= add_months(date_trunc('MONTH', current_date()), -2)
    AND usage_date < date_trunc('MONTH', current_date())
  GROUP BY ALL
)
SELECT currency_code, product,
  round(sum(CASE WHEN month_start = add_months(date_trunc('MONTH', current_date()), -2) THEN cost ELSE 0 END), 2) AS earlier_month,
  round(sum(CASE WHEN month_start = add_months(date_trunc('MONTH', current_date()), -1) THEN cost ELSE 0 END), 2) AS previous_month,
  round(100.0 * (previous_month - earlier_month) / nullif(earlier_month, 0), 2) AS growth_pct
FROM monthly
GROUP BY currency_code, product
ORDER BY growth_pct DESC;

## 10. Alerts and control queries

A Databricks SQL alert should return rows only when action is required. Avoid alerting on the current partial day because usage data arrives with delay. Configure the query schedule, owner, notification destination, and runbook outside this notebook.

In [ ]:
%sql
SELECT usage_date, currency_code, round(sum(estimated_list_cost), 2) AS daily_cost,
       CAST('${daily_cost_alert}' AS DECIMAL(18,2)) AS threshold
FROM billing_cost_enriched
WHERE usage_date = date_sub(current_date(), 1)
GROUP BY usage_date, currency_code
HAVING daily_cost > threshold;

In [ ]:
%sql
-- Data-quality/control failures that should be investigated.
SELECT 'MISSING_PRICE' AS control, count_if(missing_price) AS failures FROM billing_cost_enriched
UNION ALL
SELECT 'MISSING_IDENTITY', count_if(identity_metadata.run_as IS NULL) FROM billing_cost_enriched
UNION ALL
SELECT 'MISSING_ALLOCATION_TAG', count_if(element_at(custom_tags, '${tag_key}') IS NULL) FROM billing_cost_enriched
UNION ALL
SELECT 'FUTURE_USAGE_DATE', count_if(usage_date > current_date()) FROM billing_cost_enriched;

## 11. Production Gold-table pattern

For dashboards, materialize a governed daily aggregate rather than repeatedly scanning raw billing data. Incrementally recompute a rolling window because corrections can arrive after the original usage date. Preserve dimensions required for reconciliation, and restrict access to identity-level detail.

Example grain: `usage_date, workspace_id, product, sku_name, currency_code, cost_center, resource_type, resource_id`.

Recommended controls:

1. Reprocess at least the recent correction window on every run.
2. Reconcile Gold cost to the correction-aware source by date and currency.
3. Monitor missing price, identity, resource, and allocation-tag coverage.
4. Store policy version and allocation method for shared-cost chargeback.
5. Apply row filters or separate views if teams may see only their own cost centers.
6. Record dashboard/alert ownership and a response runbook.

## 12. Advanced exercises

1. Build an executive dashboard with MTD cost, forecast, product mix, top workspaces, top jobs, and untagged-cost percentage.
2. Create an allocation policy table and split shared platform cost by each team's directly attributed cost. Explain why the method is fair or unfair.
3. Join `system.lakeflow.job_run_timeline` and calculate cost and runtime per successful job run. Identify expensive retries and failed runs.
4. Join cluster and node timeline tables to estimate idle classic-compute cost. State all assumptions.
5. Use warehouse events and query history to find warehouses with billed usage but little or no query activity.
6. Calculate cost per business unit of work, such as cost per million source records or per successful pipeline update. This requires a trusted denominator table.
7. Compare Photon and non-Photon workloads by runtime and total cost; do not assume a higher SKU price means a higher total workload cost.
8. Implement a 30-day rolling Gold table and prove it reconciles with raw billing records after a correction.
9. Design budget thresholds at account, workspace, product, and cost-center levels with warning and critical severity.
10. Write a one-page FinOps runbook: alert triage, responsible owner, evidence to collect, safe actions, and escalation path.

## 13. Interpretation checklist

Before publishing a number, answer all of these:

- Is it usage, estimated list cost, negotiated platform cost, cloud infrastructure cost, or invoice cost?
- Are corrections included?
- Was the historical price effective at usage time used?
- Are currencies and units kept separate?
- Is the current partial day/month excluded where appropriate?
- How much cost lacks a price, identity, resource ID, or allocation tag?
- Is the source global or regional, and what is its retention/freshness?
- Is an allocation exact or approximate?
- Can the result reconcile back to `system.billing.usage`?
- Does an alert have an owner and an actionable runbook?

## Official references

- [Billable usage system table](https://docs.databricks.com/aws/en/admin/system-tables/billing)
- [List price system table](https://docs.databricks.com/aws/en/admin/system-tables/pricing)
- [Monitor costs using system tables](https://docs.databricks.com/aws/en/admin/usage/system-tables)
- [Jobs system tables and cost attribution](https://docs.databricks.com/aws/en/admin/system-tables/jobs)
- [Compute system tables](https://docs.databricks.com/aws/en/admin/system-tables/compute)
- [SQL warehouse monitoring queries](https://docs.databricks.com/aws/en/compute/sql-warehouse/monitor/queries)
- [System tables reference](https://docs.databricks.com/aws/en/admin/system-tables/)